# 06 — Demo del pipeline end-to-end

Carga los modelos entrenados (pants y tops) y procesa un puñado de imágenes
de prueba, mostrando segmentación + paleta de colores + predicción
de atributos.


## 1. Setup

In [ ]:
from google.colab import auth
auth.authenticate_user()
print('Autenticación exitosa.')

BUCKET     = 'gs://jbj-vision'
REPO_DIR   = '/content/repo'
CACHE_ROOT = '/content/cache'

# Sincronizar repo desde GCS
!gsutil -m rsync -r -x '\.git/.*|__pycache__/.*' {BUCKET}/code/ {REPO_DIR}
%cd {REPO_DIR}
!pip install -q gcsfs==2023.10.0 fsspec google-cloud-storage
!pip install -q -r requirements.txt


## 1.5. Restaurar cache de imágenes desde GCS

In [ ]:
import subprocess, os

def pull_tarball(ds: str) -> bool:
    gcs_path = f"{BUCKET}/data/processed/images_{ds}.tar.gz"
    local_tar = f"/tmp/images_{ds}.tar.gz"
    r = subprocess.run(['gsutil', 'ls', gcs_path], capture_output=True)
    if r.returncode == 0:
        print(f'Pulling {gcs_path} ...')
        os.makedirs(config['paths'][f'images_{ds}'], exist_ok=True)
        !gsutil -m cp {gcs_path} {local_tar}
        !tar xzf {local_tar} -C {config['paths'][f'images_{ds}']} --strip-components=1
        print(f'✓ Cache {ds} restaurado.')
        return True
    print(f'Sin tarball para {ds} — ejecutá 02_prepare_data_colab.ipynb primero.')
    return False

pull_tarball('pants')
pull_tarball('tops')


## 2. Inicializar pipeline

In [ ]:
import sys
sys.path.insert(0, '/content/repo')

from src.data.colab import setup_gcp
config = setup_gcp('config/pipeline_config.yaml', bucket='gs://jbj-vision', local_cache_root='/content/cache')
print('Checkpoints:')
print('  pants:', config['pants']['checkpoint'])
print('  tops: ', config['tops']['checkpoint'])

import yaml
patched_config_path = '/tmp/config_patched.yaml'
with open(patched_config_path, 'w') as f:
    yaml.safe_dump(config, f)

from src.pipeline import FashionPipeline
pipeline = FashionPipeline(patched_config_path)


## 3. Procesar imágenes de prueba

In [ ]:
# Tomamos imágenes del cache local (restaurado arriba)
import random, json
from pathlib import Path

random.seed(42)
pants_imgs = list(Path(config['paths']['images_pants']).glob('*.jpg'))[:3]
tops_imgs  = list(Path(config['paths']['images_tops']).glob('*.jpg'))[:3]
sample = random.sample(pants_imgs + tops_imgs, min(6, len(pants_imgs) + len(tops_imgs)))

if not sample:
    print('⚠ No hay imágenes en cache local. Ejecutá 02_prepare_data_colab.ipynb primero.')
else:
    results = pipeline.process_batch(sample)
    for r in results:
        print(json.dumps(r, indent=2, ensure_ascii=False)[:400], '\n---')


## 4. Visualización

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

n = len(results)
fig, axes = plt.subplots(1, n, figsize=(4 * n, 5))
if n == 1: axes = [axes]
for ax, r in zip(axes, results):
    img = Image.open(r['image_path'])
    ax.imshow(img); ax.axis('off')
    title = f"tipo: {r.get('garment_type', '?')}\n"
    attrs = r.get('attributes', {})
    for k, v in list(attrs.items())[:3]:
        title += f"{k}: {v.get('label', '?')} ({v.get('confidence', 0):.2f})\n"
    ax.set_title(title, fontsize=9)
plt.tight_layout(); plt.show()

## 5. Evaluación contra ground truth (opcional)

In [ ]:
# Si las imagenes vienen de los CSV, podemos comparar con ground truth
import pandas as pd

# Cargar CSVs
df_pants = pd.read_csv(config['data']['pants_csv'])
df_tops = pd.read_csv(config['data']['tops_csv'])

# Para cada result intentamos encontrar el id en los CSV
# (asume que las imagenes tienen el id en el nombre)
print('Para evaluacion completa, ejecuta:')
print('  python scripts/evaluate.py --predictions outputs/results.json \\')
print('         --csv data/preprocessed/pants_1.csv --dataset pants')
